In [1]:
import os
import json
import shutil
import sys

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED, init_plsa

In [4]:
import artm
from artm import ARTM, Dictionary

import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

from topicnet.cooking_machine.models.topic_model import ARTM_NINE
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.viewers.top_documents_viewer import TopDocumentsViewer
from topicnet.viewers.top_tokens_viewer import TopTokensViewer
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    count_vocab_size,
    init_model,
)
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    modality_weight_rel2abs,
    transform_regularizer,
)


import numpy as np
import pandas as pd
from pandas import DataFrame
from scipy.spatial.distance import cdist

import os
import tempfile
import warnings
from copy import deepcopy
from typing import Dict, List, Optional

In [5]:
BERTOPIC_FOLDER_PATH = '/data_mil/shared/CompressaAI/BERTopic'
RESULTS_FOLDER_PATH = os.path.join(BERTOPIC_FOLDER_PATH, 'results', '20newsgroups')

In [6]:
! ls $RESULTS_FOLDER_PATH

0  1  2  3  4


In [7]:
! ls $RESULTS_FOLDER_PATH/0

dataset.csv  dataset__internals  phi.csv  top_words.json


In [8]:
dataset = Dataset(
    f'{RESULTS_FOLDER_PATH}/0/dataset.csv',
)

dataset.get_possible_modalities()

{'@lemmatized'}

In [9]:
MAIN_MODALITY = '@lemmatized'

In [10]:
dataset._data.head()

,Unnamed: 0,id,vw_text
id,,,
rec_autos_102994,0,rec_autos_102994,rec_autos_102994 |@lemmatized was wondering if...
comp_sys_mac_hardware_51861,1,comp_sys_mac_hardware_51861,comp_sys_mac_hardware_51861 |@lemmatized fair ...
comp_sys_mac_hardware_51879,2,comp_sys_mac_hardware_51879,comp_sys_mac_hardware_51879 |@lemmatized well ...
comp_graphics_38242,3,comp_graphics_38242,comp_graphics_38242 |@lemmatized Do you have W...
sci_space_60880,4,sci_space_60880,sci_space_60880 |@lemmatized From article C5ow...


In [11]:
phi0 = pd.read_csv(f'{RESULTS_FOLDER_PATH}/0/phi.csv', index_col=0)

In [12]:
phi0.head()

,background_1,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,...,topic_10,topic_11,topic_12,topic_13,topic_14,topic_15,topic_16,topic_17,topic_18,topic_19
00,0.0,0.001167,0.000092,0.005393,0.0,0.001362,0.000000,0.000000,0.0,0.0,...,0.000737,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
000,0.0,0.000270,0.000051,0.004587,0.0,0.000254,0.000151,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0000,0.0,0.000050,0.000000,0.003186,0.0,0.000000,0.000000,0.000121,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
00000,0.0,0.000039,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
000000,0.0,0.000126,0.000000,0.000000,0.0,0.000473,0.000000,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
phi0.shape

(97541, 21)

In [14]:
phi0.set_index([[MAIN_MODALITY] * len(phi0.index), phi0.index], inplace=True)

In [15]:
phi0.sum(axis=0)

background_1    6.419650
topic_0         6.345205
topic_1         6.062952
topic_2         6.507537
topic_3         6.308676
topic_4         6.537928
topic_5         6.431884
topic_6         6.305140
topic_7         6.767294
topic_8         5.938382
topic_9         5.970806
topic_10        6.570667
topic_11        6.863978
topic_12        5.966615
topic_13        6.075575
topic_14        6.103501
topic_15        5.832443
topic_16        6.409397
topic_17        6.341861
topic_18        5.961218
topic_19        5.698086
dtype: float64

In [16]:
min(phi0.sum(axis=0)), max(phi0.sum(axis=0))

(5.698085908033937, 6.8639784215610575)

In [17]:
dictionary = dataset.get_dictionary()

In [18]:
dictionary

artm.Dictionary(name=5e94fbf5-85b7-46b0-bf46-29cfb34b910f, num_entries=114951)

In [19]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [20]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 6.77 s, sys: 312 ms, total: 7.08 s
Wall time: 7 s


In [21]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [22]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [23]:
def view_model(
        topic_model,
        dataset,
        num_top_tokens: int = 5,
        top_tokens_method: str = 'phi',
        num_topics: Optional[int] = 5,  # we do not want to fill the whole .ipynb notebook with topics...
        ):
    top_tok_viewer = TopTokensViewer(
        topic_model, num_top_tokens=num_top_tokens, method=top_tokens_method
    )
    top_doc_viewer = TopDocumentsViewer(topic_model, dataset=dataset)
    top_docs = top_doc_viewer.view()

    if num_topics is None:
        num_topics = len(topic_model.topic_names)

    for topic_name in topic_model.topic_names[:num_topics]:
        topic_top_toks = top_tok_viewer.to_html(topic_names=[topic_name])
        topic_top_docs = top_docs[topic_name]
        display_html(topic_top_toks, raw=True)
        display(topic_top_docs)

In [24]:
class FastFixPhiRegularizer(BaseRegularizer):
    _VERY_BIG_TAU = 10 ** 9

    def __init__(self, name: str, topic_names: List[str], parent_model=None, parent_phi=None):
        super().__init__(name, tau=self._VERY_BIG_TAU)

        self._topic_names = topic_names
        self._topic_indices = None
        self._parent_model = parent_model
        self._parent_phi = parent_phi

    def grad(self, pwt, nwt):
        # print('Fixing')

        rwt = np.zeros_like(pwt)

        if self._parent_phi is not None:
            parent_phi = self._parent_phi
            vals = parent_phi.values
        else:
            parent_phi = self._parent_model.get_phi()
            vals = parent_phi.values[:, self._topic_indices]

        assert vals.shape[0] == rwt.shape[0]
        assert vals.shape[1] == len(self._topic_indices)
        
        rwt[:, self._topic_indices] += vals

        return self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = self._model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

In [25]:
NUM_TOPICS = 20
NUM_ITERATIONS = 5
NUM_TOP_TOKENS = 20

In [26]:
NUM_TOPICS

20

In [47]:
def fit_and_compute_scores(model, dataset, target_topic_indices=None, custom_regularizers=None):
    print(custom_regularizers)

    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS, custom_regularizers=custom_regularizers)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account

    if target_topic_indices is None:
        target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()

    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}

    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
    }

In [28]:
def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [29]:
NUM_TRAINS = 3
TOPIC_INDICES = list(range(NUM_TOPICS))

In [30]:
TOPIC_INDICES

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

In [31]:
NUM_TOPICS

20

In [32]:
model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=NUM_TOPICS + 1,  # background
    seed=0
)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



In [33]:
model.get_phi().shape

(114951, 21)

In [34]:
phi = model.get_phi()

In [35]:
phi.columns

Index(['topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5',
       'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10', 'topic_11',
       'topic_12', 'topic_13', 'topic_14', 'topic_15', 'topic_16', 'topic_17',
       'topic_18', 'topic_19', 'topic_20'],
      dtype='object')

In [36]:
assert phi.shape[1] == phi0.shape[1], (phi.shape[1], phi0.shape[1])

In [37]:
common_words = list(set(phi.index).intersection(phi0.index))

phi.loc[:, :] = 0
phi.loc[common_words,:] += phi0.loc[common_words,:]

phi = phi / phi.sum(axis=0)

In [38]:
phi.shape

(114951, 21)

In [39]:
with open(f'{RESULTS_FOLDER_PATH}/0/top_words.json', 'r') as f:
    top_words = json.loads(f.read())

In [40]:
top_words.keys()

dict_keys(['topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5', 'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10', 'topic_11', 'topic_12', 'topic_13', 'topic_14', 'topic_15', 'topic_16', 'topic_17', 'topic_18', 'topic_19'])

In [43]:
DIFF_THRESHOLD = 2

In [44]:
for t, topic_top_words in top_words.items():
    print(t)
    # print(top_words)

    top_phi = set(phi[t].sort_values(ascending=False)[:NUM_TOP_TOKENS].index.get_level_values(1))
    top_bt = set([p[0] for p in topic_top_words])

    if top_phi != top_bt:
        diff1 = top_phi.difference(top_bt)
        diff2 = top_bt.difference(top_phi)

        print('  WTF:', diff1, diff2)

        if len(diff1) > DIFF_THRESHOLD:
            print(f'  WTF?!?!?', len(diff1))

        if len(diff2) > DIFF_THRESHOLD:
            print(f'  WTF?!?!?', len(diff2))
        

# Whatever...

topic_0
topic_1
topic_2
  WTF: {'good'} {'nhl'}
topic_3
topic_4
topic_5
  WTF: {'doctors', 'banks'} {'gordon', 'n3jxp'}
topic_6
  WTF: {'dont', 'ottoman', 'saw', 'government', 'know', 'started'} {'armenian', 'turks', 'sumgait', 'azerbaijan', 'armenians', 'armenia'}
  WTF?!?!? 6
  WTF?!?!? 6
topic_7
  WTF: {'L2PMABGZ7VAZV0PZRI', 'NikeCajun', '207556000', 'effortsall', 'knowlege', '8800CS', 'CDROMCATZIP', 'dragdrop', 'toolbox', 'ringleaders', 'CSCSTD00385', '1795', 'taxation', '5152940082', 'Epilepsy'} {''}
  WTF?!?!? 15
topic_8
topic_9
  WTF: {'social'} {'lsd'}
topic_10
topic_11
  WTF: {'vol', 'writing', 'molecular', 'manual'} {'vernor', 'vinge', 'baen', 'gibson'}
  WTF?!?!? 4
  WTF?!?!? 4
topic_12
  WTF: {'king', 'forget', 'computer', 'rule', 'guide'} {'douglas', 'adams', 'altima', 'alice', 'infiniti'}
  WTF?!?!? 5
  WTF?!?!? 5
topic_13
  WTF: {'students', 'andrew', 'pm', 'cs'} {'jstmp', 'carnegie', 'mellon', 'japan'}
  WTF?!?!? 4
  WTF?!?!? 4
topic_14
topic_15
  WTF: {'stripped', 'reg

In [48]:
phi.columns

Index(['topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5',
       'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10', 'topic_11',
       'topic_12', 'topic_13', 'topic_14', 'topic_15', 'topic_16', 'topic_17',
       'topic_18', 'topic_19', 'topic_20'],
      dtype='object')

In [51]:
fix_regularizer = FastFixPhiRegularizer(
    name='fix',
    parent_phi=phi.iloc[:, 1:],
    topic_names=phi.columns[1:],
)

custom_regularizers = {
    fix_regularizer.name: fix_regularizer,
}

result = fit_and_compute_scores(
    model, dataset,
    target_topic_indices=list(range(phi.shape[1])),
    custom_regularizers=custom_regularizers
)

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f992830dee0>}


In [53]:
result

{'scores': {'perplexity': 2334.455078125,
  'coherence_20': array([1.4539663]),
  'diversity_euclidean': 0.11831513976417281,
  'diversity_jensenshannon': 0.7622861942596016,
  'diversity_hellinger': 0.882083585763223,
  'diversity_cosine': 0.8845350885792939},
 'topic_coherences': {0: 0.2332319384114421,
  1: 0.7828129948159336,
  2: 1.6570045095465096,
  3: 1.0912535311363014,
  4: 1.9160995627076196,
  5: 1.515770697307198,
  6: 1.0234783229373245,
  7: 0.05614310769281872,
  8: 1.5587540470905896,
  9: 2.1618052278586286,
  10: 1.7600162217628093,
  11: 1.3728873168770976,
  12: 1.5945788250399215,
  13: 1.3113348631878727,
  14: 3.7567500660203463,
  15: 1.3770167995683047,
  16: 2.3764950147175274,
  17: 2.150769177059853,
  18: 1.1946909515501167,
  19: 1.6025324815271995,
  20: 0.03986670434526436}}

In [54]:
result['scores']

{'perplexity': 2334.455078125,
 'coherence_20': array([1.4539663]),
 'diversity_euclidean': 0.11831513976417281,
 'diversity_jensenshannon': 0.7622861942596016,
 'diversity_hellinger': 0.882083585763223,
 'diversity_cosine': 0.8845350885792939}

In [55]:
cheatty_ppl = result['scores']['perplexity']

In [56]:
cheatty_ppl

2334.455078125

In [57]:
model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=NUM_TOPICS,
    seed=0
)

phi = model.get_phi()

assert phi.shape[1] == phi0.shape[1] - 1

common_words = list(set(phi.index).intersection(phi0.index))

phi.loc[:, :] = 0
phi.loc[common_words, :] += phi0.loc[common_words, phi.columns]

assert set(phi0.columns).difference(set(phi.columns)) == {'background_1'}

phi = phi / phi.sum(axis=0)

fix_regularizer = FastFixPhiRegularizer(
    name='fix',
    parent_phi=phi,
    topic_names=phi.columns,
)
custom_regularizers = {
    fix_regularizer.name: fix_regularizer,
}

result = fit_and_compute_scores(model, dataset, custom_regularizers=custom_regularizers)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9928fafdf0>}


In [58]:
result

{'scores': {'perplexity': 116999.8359375,
  'coherence_20': array([1.59646966]),
  'diversity_euclidean': 0.11651928466902746,
  'diversity_jensenshannon': 0.7540381700256178,
  'diversity_hellinger': 0.8895890345296852,
  'diversity_cosine': 0.8593474498289518},
 'topic_coherences': {0: 0.8065581739518853,
  1: 0.7828129948159336,
  2: 1.6570045095465096,
  3: 1.0912535311363014,
  4: 1.9160995627076196,
  5: 1.515770697307198,
  6: 1.0234783229373245,
  7: 0.9326767872324533,
  8: 1.5587540470905896,
  9: 2.1618052278586286,
  10: 1.7600162217628093,
  11: 1.3728873168770976,
  12: 1.5945788250399215,
  13: 1.3113348631878727,
  14: 3.7567500660203463,
  15: 1.3770167995683051,
  16: 2.3764950147175274,
  17: 2.136876804854279,
  18: 1.1946909515501167,
  19: 1.6025324815271995}}

In [59]:
result['scores']['cheatty_ppl'] = cheatty_ppl
result['scores']['fair_ppl'] = result['scores']['perplexity']

In [60]:
result

{'scores': {'perplexity': 116999.8359375,
  'coherence_20': array([1.59646966]),
  'diversity_euclidean': 0.11651928466902746,
  'diversity_jensenshannon': 0.7540381700256178,
  'diversity_hellinger': 0.8895890345296852,
  'diversity_cosine': 0.8593474498289518,
  'cheatty_ppl': 2334.455078125,
  'fair_ppl': 116999.8359375},
 'topic_coherences': {0: 0.8065581739518853,
  1: 0.7828129948159336,
  2: 1.6570045095465096,
  3: 1.0912535311363014,
  4: 1.9160995627076196,
  5: 1.515770697307198,
  6: 1.0234783229373245,
  7: 0.9326767872324533,
  8: 1.5587540470905896,
  9: 2.1618052278586286,
  10: 1.7600162217628093,
  11: 1.3728873168770976,
  12: 1.5945788250399215,
  13: 1.3113348631878727,
  14: 3.7567500660203463,
  15: 1.3770167995683051,
  16: 2.3764950147175274,
  17: 2.136876804854279,
  18: 1.1946909515501167,
  19: 1.6025324815271995}}

In [62]:
results = dict()

results[0] = result

In [63]:
for k, r in results.items():
    r['scores']['coherence_20'] = float(r['scores']['coherence_20'])

In [64]:
results

{0: {'scores': {'perplexity': 116999.8359375,
   'coherence_20': 1.5964696599844959,
   'diversity_euclidean': 0.11651928466902746,
   'diversity_jensenshannon': 0.7540381700256178,
   'diversity_hellinger': 0.8895890345296852,
   'diversity_cosine': 0.8593474498289518,
   'cheatty_ppl': 2334.455078125,
   'fair_ppl': 116999.8359375},
  'topic_coherences': {0: 0.8065581739518853,
   1: 0.7828129948159336,
   2: 1.6570045095465096,
   3: 1.0912535311363014,
   4: 1.9160995627076196,
   5: 1.515770697307198,
   6: 1.0234783229373245,
   7: 0.9326767872324533,
   8: 1.5587540470905896,
   9: 2.1618052278586286,
   10: 1.7600162217628093,
   11: 1.3728873168770976,
   12: 1.5945788250399215,
   13: 1.3113348631878727,
   14: 3.7567500660203463,
   15: 1.3770167995683051,
   16: 2.3764950147175274,
   17: 2.136876804854279,
   18: 1.1946909515501167,
   19: 1.6025324815271995}}}

In [ ]:
for k, r in results.items():
    with open(f'test_bt_results_{k}.json', 'w') as f:
        f.write(
            json.dumps(r, indent=4)
        )

In [128]:
('@word', 'nsa') in model.get_phi().index

False

In [127]:
for d in dataset._data['vw_text']:
    if 'zq6kkf8hkjoj5jcwcfper6j' in d:
        print(d)

In [59]:
phi0.iloc[:, 0] > 0

@word  00          True
       000         True
       0000        True
       00000      False
       000000     False
                  ...  
       zz960      False
       zzc2       False
       zzzs       False
       zzzzzz      True
       zzzzzzt     True
Name: topic_0, Length: 97851, dtype: bool

In [99]:
phi = model.get_phi()

In [100]:
phi0.sum(axis=0)

topic_0      3.563466
topic_1      3.105079
topic_2      2.712872
topic_3      2.873670
topic_4      2.740778
               ...   
topic_141    4.756303
topic_142    3.229138
topic_143    2.920812
topic_144    3.076361
topic_145    2.709027
Length: 146, dtype: float64

In [101]:
phi.shape[1], phi0.shape[1]

(146, 146)

In [102]:
common_words = list(set(phi.index).intersection(phi0.index))

In [103]:
phi.loc[:, :] = 0

In [104]:
phi.sum(axis=0)

topic_0      0.0
topic_1      0.0
topic_2      0.0
topic_3      0.0
topic_4      0.0
            ... 
topic_141    0.0
topic_142    0.0
topic_143    0.0
topic_144    0.0
topic_145    0.0
Length: 146, dtype: float32

In [105]:
phi0.columns

Index(['topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5',
       'topic_6', 'topic_7', 'topic_8', 'topic_9',
       ...
       'topic_136', 'topic_137', 'topic_138', 'topic_139', 'topic_140',
       'topic_141', 'topic_142', 'topic_143', 'topic_144', 'topic_145'],
      dtype='object', length=146)

In [106]:
phi.loc[common_words,:] += phi0.loc[common_words,:]

In [107]:
phi.sum(axis=0)

topic_0      2.803020
topic_1      2.900242
topic_2      2.579136
topic_3      2.616285
topic_4      2.573774
               ...   
topic_141    3.882989
topic_142    3.013007
topic_143    2.576366
topic_144    2.883666
topic_145    2.496880
Length: 146, dtype: float64

In [108]:
min(phi.sum(axis=0)), max(phi.sum(axis=0))

(2.2481894708336334, 3.8829893193144196)

In [109]:
phi = phi / phi.sum(axis=0)

In [129]:
phi['topic_0'].sort_values(ascending=False)[:10]

modality  token  
@word     team       0.003889
          game       0.003540
          he         0.003227
          season     0.002869
          games      0.002801
          players    0.002687
          play       0.002677
          hockey     0.002592
          year       0.002399
          league     0.002248
Name: topic_0, dtype: float64

In [122]:
set(phi['topic_0'].sort_values(ascending=False)[:10].index.get_level_values(1))

{'game',
 'games',
 'he',
 'hockey',
 'league',
 'play',
 'players',
 'season',
 'team',
 'year'}

In [114]:
with open(DATA_FOLDER_PATH + '/test_bt_topwords.json', 'r') as f:
    bt_topwords = json.loads(f.read())

In [131]:
for t, top_words in bt_topwords.items():
    print(t)
    # print(top_words)

    top_phi = set(phi[t].sort_values(ascending=False)[:NUM_TOP_TOKENS].index.get_level_values(1))
    top_bt = set([p[0] for p in top_words])

    if top_phi != top_bt:
        print(top_phi.difference(top_bt), top_bt.difference(top_phi))

# Whatever...

topic_0
topic_1
topic_2
{'is'} {'nsa'}
topic_3
topic_4
{'have'} {'fbi'}
topic_5
topic_6
topic_7
topic_8
{'that'} {'ssf'}
topic_9
topic_10
topic_11
{'users', 'of'} {'o157h7', 'hus'}
topic_12
topic_13
topic_14
topic_15
{'program'} {'jfif'}
topic_16
topic_17
{'code'} {'null'}
topic_18
{'typinginjuryfaqgeneral', 'Neuhaus', 'scoresheets', 'attacks', 'Ellen', 'ecclesisatical', 'GrayHound', 'wraptype', 'OConnor', 'antibiotic', 'directors', '437', '258bit'} {''}
topic_19
topic_20
topic_21
{'it'} {'bj200'}
topic_22
topic_23
topic_24
{'gaybi', 'as', 'dont'} {'enviroleague', 'cramer', 'bsa'}
topic_25
{'size'} {'mathcad'}
topic_26
topic_27
topic_28
{'university', 'history', 'they', 'had', 'population'} {'turks', 'armenian', 'armenians', 'armenia', 'argic'}
topic_29
{'book', 'as', 'prophets'} {'quran', 'muhammad', 'rushdie'}
topic_30
topic_31
{'card', 'devices'} {'scsi2', 'scsi1'}
topic_32
{'duo'} {'irqs'}
topic_33
{'adventure', 'interested'} {'snes', 'sega'}
topic_34
topic_35
{'can', 'provides'} {

In [133]:
fix_regularizer = FastFixPhiRegularizer(
    name='fix',
    parent_phi=phi,
    topic_names=phi.columns,
)

custom_regularizers = {
    fix_regularizer.name: fix_regularizer,
}

result = fit_and_compute_scores(model, dataset, custom_regularizers=custom_regularizers)

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f71851de850>}


In [134]:
result = new_result

In [136]:
result['scores']

{'perplexity': 9313.6455078125,
 'coherence_20': array([1.53849689]),
 'diversity_euclidean': 0.06944713960080998,
 'diversity_jensenshannon': 0.7210854989101421,
 'diversity_hellinger': 0.8565287662161661,
 'diversity_cosine': 0.8326357482944967}

In [140]:
results = {
    'test': result
}

In [143]:
r

{'scores': {'perplexity': 9313.6455078125,
  'coherence_20': array([1.53849689]),
  'diversity_euclidean': 0.06944713960080998,
  'diversity_jensenshannon': 0.7210854989101421,
  'diversity_hellinger': 0.8565287662161661,
  'diversity_cosine': 0.8326357482944967},
 'topic_coherences': {0: 1.2252953493262035,
  1: 0.622924615224085,
  2: 1.1583919440035115,
  3: 1.0855657296738097,
  4: 0.677690967785305,
  5: 0.5630280933439489,
  6: 1.2178763350206647,
  7: 1.31892096280969,
  8: 1.0080110613759343,
  9: 1.726066996458189,
  10: 1.411324251030534,
  11: 1.6803373416371612,
  12: 0.74318595857547,
  13: 2.249348135503753,
  14: 0.9931107411332841,
  15: 1.799890990293903,
  16: 1.0286068610126915,
  17: 1.6705548497219567,
  18: 0.3213578724353934,
  19: 1.235485065372899,
  20: 1.566994324245821,
  21: 1.9020915201357333,
  22: 0.9738945194022423,
  23: 1.1826511243101272,
  24: 2.2424500177628186,
  25: 1.439606731445224,
  26: 1.411553817644481,
  27: 1.0335294683808687,
  28: 0.676

In [144]:
for k, r in results.items():
    r['scores']['coherence_20'] = float(r['scores']['coherence_20'])

for k, r in results.items():
    with open(f'test_bt_results_{k}.json', 'w') as f:
        f.write(
            json.dumps(r, indent=4)
        )